# Detector retraining (Stage 3): EX and HE

**Task.** Retrain the exudate (EX) and haemorrhage (HE) detectors with two changes the Stage 1/2 evaluation pointed to: photos resized so the fundus has one common size, and healthy eyes as negatives. Choose the epoch on validation data only; compare once with the current detectors on the locked test sets. Started 2026-11-05 by Victoria's decision, before Adam's sign-off; the candidates replace nothing until he agrees and the test comparison is done.

**How to use.** Run All. Settings live in the Parameters cell; on the pod `RETRAIN_DETS=ex` or `he` runs one detector (two processes can share the GPU). Outputs are overwritten in `notebooks/outputs/detector_retraining/<det>/`; a finished training is not repeated (delete `<det>/train_log.csv` to retrain).

**Rules.** No hand-typed results in markdown; every reported figure comes from the generated Claims table. Hard checks stop the notebook: file MD5s; checkpoint identities; the validation numbers of the current detectors (Stage 2 record) must reproduce with this notebook's code before any training counts; three independent matching paths must agree on the chosen model.

## Pre-specified design, questions and decision rules

**Data** (`inputs/detector_retrain_manifest.csv`, fixed by `detector_retraining_prep.ipynb`): train = IDRiD-train 54, DDR train 383 + valid 149, EyePACS grade 0 train partition 600 (one eye per patient, fellow-eye-DR and val/test patients excluded); val = DDR-test validation half 114 + EyePACS dev 150; test = DDR-test test half 111 + IDRiD-test 27 + EyePACS holdout 300.

**Per detector, training images are:** *positives* (grader mask of that type not blank); *outlined negatives* (DDR images whose mask of that type is blank, kept after the screen below); *healthy negatives* (EyePACS; each one enters an epoch with probability equal to its manifest weight, 0.5). The IDRiD-train image without an HE mask file is not used for HE. Prep found no grader pixels outside the fundus in any training mask, so the masks are used as graded.

**Blank-mask screen (rule fixed before looking).** A blank DDR mask is not proof of no lesion (007-1974-100 has a yellow cluster and a blank EX mask). Each blank-mask DDR training image is run through the current detector (production configuration, 512 level). It is left out of that detector's negatives if its detected area is above the 95th percentile of the false-positive area of the current detector on the 150 validation healthy eyes. Left-out images are listed and the largest are shown.

**What stays as in the current detectors** (`train_ex_detector.py` @ victoria-ex-seed-check, `train_he_detector.py` @ victoria-he-seed-check-v2): U-Net + EfficientNet-B2 (ImageNet start); CLAHE on LAB L; Focal Tversky + Dice loss (EX alpha 0.6, HE 0.7, gamma 1.33); AdamW lr 1e-4 (weight decay EX 1e-3, HE 1e-4), cosine schedule over 80 epochs; patches EX 512 (6 lesion-centred + 3 random per image, batch 8), HE 768 (3 + 1, batch 4); the same augmentations; seed 42.

**What changes, and only this:** (1) every photo is resized by s = D_REF / fundus width (clipped 0.5-2.5; D_REF = IDRiD-train median, 3,416 px) before CLAHE, in training and at inference; (2) outlined and healthy negatives as above; (3) the epoch (and threshold) is chosen on the validation split, not on IDRiD-test; early stop after 4 validation checks without improvement (minimum 20 epochs). Not carried over: HE vessel-mask oversampling (no vessel masks for DDR / EyePACS; whether he_run5 used it is not in the record); `decoder_dropout=0.2` of the EX script (ignored by segmentation_models_pytorch 0.5.0: the model is identical with and without it).

**Validation measure** (identical to the detector evaluation, 512 level): inference on the resized photo, probability map resized back, EX optic-disc exclusion (od_run1_b0), any-pixel downsample to 512, fundus clip (erode 6); lesion = 4-connected grader region, found within 2 px; recall pooled over DDR-val lesions; tolerant precision pooled over DDR-val; false-positive px per healthy validation eye.

**Selection rule (validation only).** Every 5 epochs (and epoch 1), each threshold of the grid. A (epoch, threshold) qualifies if healthy false-positive px per eye is no higher than the current detector's and tolerant precision is no lower (both on validation, from the Stage 2 record). Among qualifiers: highest DDR-val recall; ties to fewer healthy false positives. If nothing qualifies by the end, no candidate goes forward and the trade-off table goes to Adam.

**Correction (2026-11-05, before any arm result counts).** The first run (now discarded, logged in decision.md) streamed patches photo by photo through a small mixing buffer and trained in mixed precision. A simulation with the real patch counts showed EX batches then held 2.6 different photos out of 8 (old scripts: 7.8), 21% of EX batches came from one photo (old: 0%), and the last 200 batches of an epoch covered 52 photos (old: 357); HE batches were mixed almost as before (3.7 of 4 vs 4.0). Both departures are removed: every patch of an epoch is collected first and shuffled together, exactly as the old scripts, and training is full precision.

**Runs (arms), one setting `RETRAIN_ARM`; fixed 2026-11-05 after the first combined run (D) did not beat the current detectors on validation.** A one-change-at-a-time design with a control, so a result can be attributed:

| Arm | Resizing | Negatives | Purpose |
|---|---|---|---|
| A (control) | no | none (lesion photos only, as the old scripts) | can this notebook reproduce the current detectors under the same selection? |
| B | yes | none | resizing alone |
| C | no | hard | hard negatives alone |
| D | yes | random | the first run (resizing + random negative patches) |
| E | yes | hard | both, only if B or C helps |

*Hard negatives* are mined once per detector, before any arm uses them: the current detector (production configuration, full resolution, threshold 0.35, EX disc excluded, fundus clip) is run on the training healthy eyes and the kept outlined-negative DDR photos; every detected blob is a hard-negative site. A negative photo's patches are centred (with the same jitter as lesion patches) on a randomly chosen site; a photo with no site gets random patches. Same number of negative patches as arm D, so C and E change *where* negatives come from, not how many. One seed per arm to screen (`RETRAIN_SEED`, 42); the best arm and the control then get seeds 123 and 7 before the single test comparison.

**Confirmation (once, test data; section 9).** Candidate vs current detector on the same images: lesion-level McNemar (exact), per-image Wilcoxon on false-positive px, McNemar on healthy any-detection; Holm across the family.

## Parameters

In [ ]:
from pathlib import Path
import os, json, time, hashlib, random, subprocess, warnings
from decimal import Decimal, ROUND_HALF_UP
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, torch, torch.nn as nn
from scipy import ndimage, stats
from skimage import measure as sk_measure
from skimage.morphology import dilation as sk_dilation
from IPython.display import display
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path(os.environ.get("RETINA_ROOT", Path.cwd().parent)).resolve(); assert (ROOT / "notebooks").is_dir(), ROOT
INPUTS = ROOT / "notebooks" / "inputs"
OUT = ROOT / "notebooks" / "outputs" / "detector_retraining"; OUT.mkdir(parents=True, exist_ok=True)
MANIFEST, MD5F, INFOF = INPUTS / "detector_retrain_manifest.csv", INPUTS / "detector_retrain_md5.csv", INPUTS / "detector_retrain_image_info.csv"

DETS = [d for d in os.environ.get("RETRAIN_DETS", "ex,he").split(",") if d]
SUBSET = int(os.environ.get("RETRAIN_SUBSET", "0"))   # quick test: N images per (split, group), 1 epoch, no record checks, no claims
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = int(os.environ.get("RETRAIN_WORKERS", "6"))
DRIVE_DIR = os.environ.get("RETRAIN_DRIVE", "gdrive:diffusion_retina_victoria/detector_retrain")   # "" = no Drive backup
ARM = os.environ.get("RETRAIN_ARM", "D")
SEED = int(os.environ.get("RETRAIN_SEED", "42"))
ARMS = {"A": dict(scale=False, neg=None), "B": dict(scale=True, neg=None), "C": dict(scale=False, neg="hard"),
        "D": dict(scale=True, neg="random"), "E": dict(scale=True, neg="hard")}
SCALED, NEG = ARMS[ARM]["scale"], ARMS[ARM]["neg"]
RUN = lambda det: f"{det}_{ARM}_s{SEED}" + ("_subset" if SUBSET else "")   # output folder per detector / arm / seed (a test run never touches a real run's folder)
TAG = f"{'_'.join(DETS)}_{ARM}_s{SEED}"
D_REF = json.loads((INPUTS / "detector_eval_meta.json").read_text())["D_REF"]; SCALE_CLIP = (0.5, 2.5)
BASE_THR = 0.35
THR_GRID = [0.2, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8]
WINDOW = {"ex": (512, 384), "he": (768, 640)}       # patch, stride (as the production detectors)
INFER_BATCH = {"ex": 8, "he": 4}
TRAIN_CFG = {   # from train_ex_detector.py / train_he_detector.py defaults
    "ex": dict(patch=512, batch=8, alpha=0.6, wd=1e-3, n_pos=6, n_neg=3),
    "he": dict(patch=768, batch=4, alpha=0.7, wd=1e-4, n_pos=3, n_neg=1)}
LR, EPOCHS, VAL_EVERY, MIN_EPOCHS, PATIENCE = 1e-4, 80, 5, 20, 4
AMP = os.environ.get("RETRAIN_AMP", "0") == "1"      # off: the old scripts trained in full precision
SCREEN_PCT = 95
CONFIRM = os.environ.get("RETRAIN_CONFIRM", "0") == "1"   # section 9: the one test comparison, after selection
# Known answers on test: Stage 1 record, current detectors (outputs/detector_evaluation/stage1_baseline.csv): (recall %, tolerant precision %) or (FP px per eye, % eyes with any)
KNOWN_TEST = {("ex", "DDR"): (73.955936, 77.250981), ("ex", "IDRiD"): (93.458647, 88.199049), ("ex", "healthy"): (93.79, 93.0),
              ("he", "DDR"): (50.077640, 69.590349), ("he", "IDRiD"): (88.235294, 64.985453), ("he", "healthy"): (513.953333, 89.0)}
# Known answers: Stage 2 record, validation split, threshold 0.35, no post-processing
# (outputs/detector_evaluation/stage2_grid_validation.csv; decision.md "Detector evaluation Stage 1 + 2 results")
KNOWN_VAL = {("ex", "native"): (72.670455, 73.037044, 160.98), ("ex", "scaled"): (80.767045, 61.000703, 213.206667),
             ("he", "native"): (49.188838, 62.660707, 305.42), ("he", "scaled"): (63.465282, 42.761247, 322.393333)}
CKPTS = {"ex": ("models/ex_detector/ex_detector_best.pth", 5, 0.7638, "0c342b8a2d"),
         "he": ("models/he_detector/he_detector_best.pth", 35, 0.7084, "76b659c647")}
OD_CKPT, OD_MD5, B2_BYTES = "models/od_detector/od_detector_best.pth", "743f1aa019", 40_673_694
TOL_PX, CLIP_ERODE = 2, 6
N_BOOT = 10_000
K_TOL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * TOL_PX + 1, 2 * TOL_PX + 1))
rhu = lambda x, nd=0: (lambda q: str(q.copy_abs() if q == 0 else q))(Decimal(str(float(x))).quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
print(ROOT, DEVICE, "dets", DETS, "arm", ARM, ARMS[ARM], "seed", SEED, "subset", SUBSET, "D_REF", D_REF)

## 1. Data and integrity checks

In [ ]:
def tissue_mask(img_rgb, thr=20):                   # verbatim from scratch/vessel_clip_proto.py
    t = (img_rgb.max(axis=2) > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(t)
    if n <= 1:
        return t > 0
    t = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    cs, _ = cv2.findContours(t, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(t); cv2.drawContours(filled, cs, -1, 1, -1)
    return filled > 0
def clip_region(img_rgb, erode=CLIP_ERODE):          # verbatim from scratch/vessel_clip_proto.py
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * erode + 1, 2 * erode + 1))
    return cv2.erode(tissue_mask(img_rgb).astype(np.uint8), k) > 0
def fov_width(bgr):
    f = 8; small = cv2.resize(bgr, (bgr.shape[1] // f, bgr.shape[0] // f), interpolation=cv2.INTER_AREA)
    xs = np.nonzero(tissue_mask(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)).any(0))[0]
    return float((xs.max() - xs.min() + 1) * f)
def read_gt(p):
    g = cv2.imread(str(ROOT / p), cv2.IMREAD_GRAYSCALE); assert g is not None, p; return g > 0

man = pd.read_csv(MANIFEST, dtype=str).fillna(""); man["weight"] = man.weight.astype(float)
assert man.name.is_unique and len(man) == 1888, len(man)
INFO = pd.read_csv(INFOF).set_index("name"); assert set(INFO.index) == set(man.name)
INFO["s"] = np.clip(D_REF / INFO.fov_w, *SCALE_CLIP)
if SUBSET:
    man = pd.concat([g.head(SUBSET) for _, g in man.groupby(["split", "origin"])]).reset_index(drop=True)
    print("SUBSET MODE:", len(man), "images; record checks and claims disabled")
missing = [p for c in ("image", "gt_ex", "gt_he") for p in man[c] if p and not (ROOT / p).exists()]
assert not missing, f"{len(missing)} files missing, e.g. {missing[:3]}"
want = pd.read_csv(MD5F).set_index("path").md5
paths = sorted({p for c in ("image", "gt_ex", "gt_he") for p in man[c] if p})
bad = [p for p in paths if hashlib.md5((ROOT / p).read_bytes()).hexdigest() != want[p]]
assert not bad, f"{len(bad)} files differ from the recorded MD5, e.g. {bad[:3]}"
print(f"MD5 verified for {len(paths)} files")
h = man[man.group == "healthy"]
for a in ("train", "val", "test"):
    for b in ("train", "val", "test"):
        if a < b: assert set(h[h.split == a].patient).isdisjoint(h[h.split == b].patient), (a, b)
assert h.groupby("split").patient.apply(lambda s: s.is_unique).all()
# fundus widths recomputed here must equal the prep notebook's (second path for the size factors)
chk = man.sample(min(40, len(man)), random_state=0)
diff = [(r.name, fov_width(cv2.imread(str(ROOT / r.image))), INFO.loc[r.name, "fov_w"]) for r in chk.itertuples()]
assert all(abs(a - b) <= 8 for _, a, b in diff), [d for d in diff if abs(d[1] - d[2]) > 8][:3]   # 8 px = one step of the 1/8 measurement
print(f"fundus width recomputed for {len(diff)} images: {sum(a == b for _, a, b in diff)} identical, all within 8 px of the prep notebook")
display(man.groupby(["split", "origin"]).size().rename("images").to_frame())

## 2. Models (identity asserted) and inference functions (as the detector evaluation)

In [ ]:
import segmentation_models_pytorch as smp
def load_prod(det):
    path, epoch, recall, md5 = CKPTS[det]; p = ROOT / path
    assert p.stat().st_size == B2_BYTES, f"{path}: not a b2 checkpoint"
    c = torch.load(p, map_location=DEVICE, weights_only=False)
    assert c["epoch"] == epoch and round(float(c["recall"]), 4) == recall, (path, c["epoch"], c["recall"])
    assert hashlib.md5(p.read_bytes()).hexdigest().startswith(md5), path
    m = smp.Unet(encoder_name="efficientnet-b2", encoder_weights=None, in_channels=3, classes=1); m.load_state_dict(c["state_dict"]); return m.to(DEVICE).eval()
oc = torch.load(ROOT / OD_CKPT, map_location=DEVICE, weights_only=False)
assert oc.get("encoder") == "efficientnet-b0" and oc.get("epoch") == 5 and round(oc.get("dice", 0), 3) == 0.963, "not od_run1_b0"
assert hashlib.md5((ROOT / OD_CKPT).read_bytes()).hexdigest().startswith(OD_MD5)
OD = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None, in_channels=3, classes=1); OD.load_state_dict(oc["state_dict"]); OD.to(DEVICE).eval(); OD_SIZE = oc["img_size"]

def preprocess(bgr):                                 # as generate_controlnet_masks.preprocess_for_detector / training
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); l, a, b = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB))
    return cv2.cvtColor(cv2.merge([cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l), a, b]), cv2.COLOR_LAB2RGB)
def starts(L, patch, stride):
    return sorted({max(0, min(s, L - patch)) for s in list(range(0, L - patch, stride)) + [L - patch]})
def infer(model, rgb, det):                          # plain-average sliding window (production)
    patch, stride = WINDOW[det]; bs = INFER_BATCH[det]
    H, W = rgb.shape[:2]; pad = ((0, max(0, patch - H)), (0, max(0, patch - W)), (0, 0))
    x = np.ascontiguousarray(np.pad(rgb, pad)); Hp, Wp = x.shape[:2]
    acc, cnt = np.zeros((Hp, Wp), np.float32), np.zeros((Hp, Wp), np.float32)
    coords = [(y, xx) for y in starts(Hp, patch, stride) for xx in starts(Wp, patch, stride)]
    with torch.no_grad():
        for i in range(0, len(coords), bs):
            cb = coords[i:i + bs]
            t = torch.from_numpy(np.stack([x[y:y + patch, xx:xx + patch] for y, xx in cb]).transpose(0, 3, 1, 2).copy()).float().div(255).to(DEVICE)
            pr = torch.sigmoid(model(t))[:, 0].float().cpu().numpy()
            for (y, xx), p in zip(cb, pr):
                acc[y:y + patch, xx:xx + patch] += p; cnt[y:y + patch, xx:xx + patch] += 1
    return np.ascontiguousarray((acc / np.maximum(cnt, 1))[:H, :W])
def od_full(bgr):                                    # as generate_controlnet_masks.od_mask_fullres
    rgb = preprocess(bgr); H, W = rgb.shape[:2]; r = cv2.resize(rgb, (OD_SIZE, OD_SIZE))
    with torch.no_grad():
        p = torch.sigmoid(OD(torch.from_numpy(r.transpose(2, 0, 1).copy()).float().unsqueeze(0).div(255).to(DEVICE)))[0, 0].cpu().numpy()
    return cv2.resize((p > 0.5).astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST) > 0
to512 = lambda m: cv2.resize(m.astype(np.uint8) * 255, (512, 512), interpolation=cv2.INTER_AREA) > 0     # any-pixel, as production
def scale_img(bgr, s):
    return cv2.resize(bgr, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA)
def prob_map(model, bgr, det, scaled, name):         # full-resolution probability at the photo's own size
    if not scaled: return infer(model, preprocess(bgr), det)
    H, W = bgr.shape[:2]
    return cv2.resize(infer(model, preprocess(scale_img(bgr, float(INFO.loc[name, "s"]))), det), (W, H), interpolation=cv2.INTER_LINEAR)
print("models and functions ready on", DEVICE)

## 3. Validation scoring (512 level; three independent matching paths)
Per validation image the 512 fundus clip, grader masks at 512 and the full-resolution optic-disc mask are computed once and cached.

In [ ]:
VCACHE = OUT / "val_cache"; VCACHE.mkdir(exist_ok=True)
VALM = man[man.split == "val"].reset_index(drop=True)
CLIP, GT512, DISC = {}, {}, {}
def load_cache(rows):
    for r in rows.itertuples():
        f = VCACHE / f"{r.name}.npz"
        if not f.exists():
            bgr = cv2.imread(str(ROOT / r.image)); b512 = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            z = dict(clip=clip_region(cv2.cvtColor(b512, cv2.COLOR_BGR2RGB)), disc=od_full(bgr))
            for det in ("ex", "he"): z[det] = to512(read_gt(getattr(r, f"gt_{det}"))) if getattr(r, f"gt_{det}") else np.zeros((512, 512), bool)
            tmp = f.parent / f"{f.stem}.{os.getpid()}.tmp.npz"; np.savez_compressed(tmp, **z); os.replace(tmp, f)   # atomic: EX and HE processes share the cache
        z = np.load(f); CLIP[r.name], DISC[r.name] = z["clip"], z["disc"]; GT512[(r.name, "ex")], GT512[(r.name, "he")] = z["ex"], z["he"]
load_cache(VALM)
LAB = {}
def gt_lab(n, det):
    if (n, det) not in LAB: LAB[(n, det)] = cv2.connectedComponents(GT512[(n, det)].astype(np.uint8), connectivity=4)
    return LAB[(n, det)]
def score_A(m, n, det):                              # cv2 path (as detector_evaluation)
    g = GT512[(n, det)]; k, lab = gt_lab(n, det); mdil = cv2.dilate(m.astype(np.uint8), K_TOL) > 0
    found = np.zeros(k, bool); found[np.unique(lab[mdil & (lab > 0)])] = True
    gd = cv2.dilate(g.astype(np.uint8), K_TOL) > 0
    return dict(lesions=k - 1, found=int(found[1:].sum()), pred_px=int(m.sum()), tp_tol=int((m & gd).sum()), fp_px=int((m & ~gd).sum())), found[1:]
def score_B(m, n, det):                              # scipy path
    g = GT512[(n, det)]; lab, k = ndimage.label(g); fk = K_TOL.astype(bool); mdil = ndimage.binary_dilation(m, structure=fk); gd = ndimage.binary_dilation(g, structure=fk)
    found = np.array([mdil[lab == i].any() for i in range(1, k + 1)], bool)
    return dict(lesions=k, found=int(found.sum()), pred_px=int(np.count_nonzero(m)), tp_tol=int(np.count_nonzero(m & gd)), fp_px=int(np.count_nonzero(m & ~gd))), found
def score_C(m, n, det):                              # skimage path
    g = GT512[(n, det)]; lab = sk_measure.label(g, connectivity=1); fk = K_TOL.astype(bool); mdil = sk_dilation(m, footprint=fk); gd = sk_dilation(g, footprint=fk)
    found = np.zeros(lab.max() + 1, bool); found[np.unique(lab[mdil & (lab > 0)])] = True
    return dict(lesions=int(lab.max()), found=int(found[1:].sum()), pred_px=int(m.sum()), tp_tol=int((m & gd).sum()), fp_px=int((m & ~gd).sum())), found[1:]
def masks512(prob, n, det, thrs=THR_GRID):           # {thr: 512 mask} with production post-processing
    if det == "ex": prob = np.where(DISC[n], 0, prob)
    return {t: to512(prob > t) & CLIP[n] for t in thrs}
def val_pass(model, det, scaled, keep=False):
    rows, kept = [], {}
    model.eval()
    for r in VALM.itertuples():
        bgr = cv2.imread(str(ROOT / r.image)); M = masks512(prob_map(model, bgr, det, scaled, r.name), r.name, det)
        for t, m in M.items(): a, _ = score_A(m, r.name, det); rows.append(dict(name=r.name, group=r.group, thr=t, **a))
        if keep: kept[r.name] = M
    return pd.DataFrame(rows), kept
def summary(df):
    out = []
    for t, d in df.groupby("thr"):
        dd, hh = d[d.group == "DDR"], d[d.group == "healthy"]
        out.append(dict(thr=t, recall_pct=100 * dd.found.sum() / max(dd.lesions.sum(), 1), prec_tol_pct=100 * dd.tp_tol.sum() / max(dd.pred_px.sum(), 1),
                        healthy_fp_px=hh.fp_px.mean(), healthy_any_pct=100 * (hh.pred_px > 0).mean()))
    return pd.DataFrame(out)
print(f"validation set: {len(VALM)} images cached")

## 4. Known answers: the current detectors on validation, native and resized (must reproduce the Stage 2 record)
This checks this notebook's inference, resizing and scoring against the detector evaluation's numbers before any training result counts.

In [ ]:
KA, KNOWN_ROWS = {}, []
for det in DETS:
    f = OUT / f"known_answers_{det}.csv"
    if f.exists(): KA[det] = pd.read_csv(f)
    else:
        m = load_prod(det); parts = []
        for mode in ("native", "scaled"):
            t0 = time.time(); df, _ = val_pass(m, det, mode == "scaled"); df["mode"] = mode; parts.append(df)
            print(det, mode, f"{time.time() - t0:.0f}s", flush=True)
        KA[det] = pd.concat(parts); del m; torch.cuda.empty_cache()
        if not SUBSET: KA[det].to_csv(f, index=False)
    for mode in ("native", "scaled"):
        s = summary(KA[det][KA[det]["mode"] == mode]).set_index("thr").loc[BASE_THR]
        rec, prec, fp = KNOWN_VAL[(det, mode)]
        KNOWN_ROWS.append(dict(det=det, mode=mode, recall_now=s.recall_pct, recall_record=rec, prec_now=s.prec_tol_pct, prec_record=prec, fp_now=s.healthy_fp_px, fp_record=fp,
                               ok=abs(s.recall_pct - rec) <= 0.3 and abs(s.prec_tol_pct - prec) <= 0.3 and abs(s.healthy_fp_px - fp) <= max(2.0, 0.02 * fp)))
KNOWN = pd.DataFrame(KNOWN_ROWS); display(KNOWN.round(3))
if not SUBSET: assert KNOWN.ok.all(), "validation record not reproduced: stop and resolve before training"
BASE = {det: KNOWN[(KNOWN.det == det) & (KNOWN["mode"] == "native")].iloc[0] for det in DETS}   # selection-rule bounds come from the RECORD values
print("known answers reproduced" if not SUBSET else "subset: known-answer check skipped")

## 5. Blank-mask screen of the DDR training negatives (rule fixed in the design)

In [ ]:
SCREEN = {}
for det in DETS:
    f = OUT / f"blank_screen_{det}.csv"
    hv = KA[det][(KA[det]["mode"] == "native") & (KA[det].thr == BASE_THR) & (KA[det].group == "healthy")].fp_px
    cutoff = float(np.percentile(hv, SCREEN_PCT))
    cand = man[(man.split == "train") & (man.group == "DDR")]
    cand = cand[[not read_gt(p).any() for p in cand[f"gt_{det}"]]]
    if f.exists(): sc = pd.read_csv(f)
    else:
        m = load_prod(det); rows = []
        for r in cand.itertuples():
            bgr = cv2.imread(str(ROOT / r.image)); p = infer(m, preprocess(bgr), det)
            if det == "ex": p = np.where(od_full(bgr), 0, p)
            b512 = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            rows.append(dict(name=r.name, det_px512=int((to512(p > BASE_THR) & clip_region(cv2.cvtColor(b512, cv2.COLOR_BGR2RGB))).sum())))
        sc = pd.DataFrame(rows); del m; torch.cuda.empty_cache()
    sc["cutoff"] = cutoff; sc["excluded"] = sc.det_px512 > cutoff; SCREEN[det] = sc
    if not SUBSET: sc.to_csv(f, index=False)
    print(f"{det.upper()}: {len(sc)} blank-mask DDR training images; cutoff {cutoff:.1f} px at 512 (P{SCREEN_PCT} of validation healthy eyes); left out {int(sc.excluded.sum())}")
    top = sc.sort_values("det_px512", ascending=False).head(8)
    if len(top):
        fig, ax = plt.subplots(2, 4, figsize=(18, 9)); ax = ax.ravel()
        for a in ax: a.axis("off")
        mm = load_prod(det)
        for a, r in zip(ax, top.itertuples()):
            img = man.set_index("name").loc[r.name, "image"]; bgr = cv2.imread(str(ROOT / img)); b512 = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            p = infer(mm, preprocess(bgr), det)
            if det == "ex": p = np.where(od_full(bgr), 0, p)
            msk = (to512(p > BASE_THR) & clip_region(cv2.cvtColor(b512, cv2.COLOR_BGR2RGB))).astype(np.uint8)
            cs, _ = cv2.findContours(msk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); vis = cv2.cvtColor(b512, cv2.COLOR_BGR2RGB).copy(); cv2.drawContours(vis, cs, -1, (0, 229, 255), 1)
            a.imshow(vis); a.set_title(f"{r.name}: {r.det_px512} px {'(left out)' if r.excluded else '(kept)'}", fontsize=9)
        del mm; torch.cuda.empty_cache()
        fig.suptitle(f"{det.upper()} blank-mask DDR training images with the largest detected area (rule-picked); cyan = current detector", fontsize=11)
        fig.tight_layout(); fig.savefig(OUT / f"blank_screen_{det}.png", dpi=100); plt.show()

## 5b. Hard-negative sites (arms C and E; mined once per detector with the current detector, shared by all arms and seeds)

In [ ]:
HARD = {}
if NEG == "hard":
    for det in DETS:
        f = OUT / f"hard_negatives_{det}.csv"
        if not f.exists():
            ex = set(SCREEN[det][SCREEN[det].excluded].name); t = man[man.split == "train"]; m = load_prod(det); rows = []
            for i, r in enumerate(t.itertuples()):
                gp = getattr(r, f"gt_{det}")
                if r.group != "healthy" and (not gp or read_gt(gp).any() or r.name in ex): continue      # negatives only: healthy + kept outlined negatives
                bgr = cv2.imread(str(ROOT / r.image)); p = infer(m, preprocess(bgr), det)
                if det == "ex": p = np.where(od_full(bgr), 0, p)
                b = ((p > BASE_THR) & clip_region(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))).astype(np.uint8)
                k, lab, st, cen = cv2.connectedComponentsWithStats(b, connectivity=8)
                rows += [dict(name=r.name, y=float(cen[j][1]), x=float(cen[j][0]), area=int(st[j, cv2.CC_STAT_AREA])) for j in range(1, k)]
                if i % 100 == 0: print(det, "mining", i, flush=True)
            h = pd.DataFrame(rows, columns=["name", "y", "x", "area"]); del m; torch.cuda.empty_cache()
            if not SUBSET: h.to_csv(f, index=False)
        else: h = pd.read_csv(f)
        HARD.update({(det, n): g[["y", "x"]].to_numpy() for n, g in h.groupby("name")})
        print(f"{det.upper()}: {len(h)} hard-negative sites in {h.name.nunique()} negative photos (median area {h.area.median() if len(h) else 0:.0f} px, full resolution)")
else: print("arm", ARM, "does not use hard negatives")

## 6. Training sets and data exploration

In [ ]:
def train_items(det):
    t = man[man.split == "train"]; ex = set(SCREEN[det][SCREEN[det].excluded].name); items = []
    for r in t.itertuples():
        gp = getattr(r, f"gt_{det}")
        if r.group == "healthy":
            if NEG: items.append(dict(name=r.name, image=r.image, gt="", kind="healthy", w=r.weight))
        elif not gp: continue                        # IDRiD-train image without a mask of this type
        elif read_gt(gp).any(): items.append(dict(name=r.name, image=r.image, gt=gp, kind="positive", w=1.0))
        elif NEG and r.name not in ex: items.append(dict(name=r.name, image=r.image, gt=gp, kind="outlined negative", w=1.0))
    return pd.DataFrame(items)
ITEMS = {det: train_items(det) for det in DETS}
for det in DETS:
    it = ITEMS[det]; it["origin"] = it.name.map(man.set_index("name").origin); it["s"] = it.name.map(INFO.s) if SCALED else 1.0
    print(det.upper()); display(it.groupby(["kind", "origin"]).agg(images=("name", "size"), median_scale=("s", "median")).round(2))
    print(f"  arm {ARM}: {int((it.kind == 'positive').sum())} lesion photos, {int((it.kind != 'positive').sum())} negative photos (the patch count per epoch is in the training log)")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for i, det in enumerate(DETS[:2]):
    it = ITEMS[det]
    for o in sorted(it.origin.unique()): ax[i].hist(it[it.origin == o].s, bins=30, alpha=0.6, label=o)
    ax[i].set(xlabel="size factor s (D_REF / fundus width)", ylabel="images", title=f"{det.upper()} training images"); ax[i].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(OUT / f"training_scale_{TAG}.png", dpi=100); plt.show()

## 7. Training (skipped when `<det>/train_log.csv` shows a finished run)

In [ ]:
import albumentations as A
from torch.utils.data import DataLoader, Dataset
class FocalTverskyLoss(nn.Module):                   # verbatim from train_he_detector.py
    def __init__(self, alpha=0.7, beta=0.3, gamma=1.33, smooth=1.0):
        super().__init__(); self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1); targets = targets.view(-1).float()
        tp = (probs * targets).sum(); fp = (probs * (1 - targets)).sum(); fn = ((1 - probs) * targets).sum()
        return (1 - (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)) ** self.gamma
class DiceLoss(nn.Module):                           # verbatim
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1); targets = targets.view(-1).float(); inter = (probs * targets).sum()
        return 1 - (2 * inter + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
def build_aug(det, patch):                           # verbatim per script
    if det == "ex":
        return A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5), A.ElasticTransform(alpha=40, sigma=6, p=0.3),
                          A.RandomBrightnessContrast(0.35, 0.35, p=0.5), A.RandomGamma(gamma_limit=(60, 140), p=0.5), A.Resize(patch, patch)])
    return A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5), A.ElasticTransform(alpha=40, sigma=6, p=0.3),
                      A.RandomBrightnessContrast(0.2, 0.2, p=0.5), A.CLAHE(clip_limit=2.0, p=0.3), A.GaussNoise(var_limit=(5, 25), p=0.2), A.Resize(patch, patch, always_apply=True)])
def _sample(image, mask, ys, xs, patch, half, n):    # verbatim (_sample_patches_from_coords)
    out = []; H, W = mask.shape
    for _ in range(n):
        i = random.randrange(len(ys))
        cy = int(np.clip(ys[i] + random.randint(-half // 2, half // 2), half, H - half)); cx = int(np.clip(xs[i] + random.randint(-half // 2, half // 2), half, W - half))
        out.append((image[cy - half:cy - half + patch, cx - half:cx - half + patch], mask[cy - half:cy - half + patch, cx - half:cx - half + patch]))
    return out
def lesion_patches(det, image, mask, patch, n_pos):  # EX: blob allocation (train_ex_detector.py); HE: any-HE pixels, no vessel masks
    half = patch // 2; out = []
    if det == "ex":
        labeled = sk_measure.label(mask > 0); regions = sk_measure.regionprops(labeled)
        allocs = [(reg, 1 if reg.area < 500 else (2 if reg.area < 5000 else 3)) for reg in regions]; scale = min(1.0, n_pos / max(sum(n for _, n in allocs), 1))
        for reg, nb in allocs:
            ys, xs = np.where(labeled == reg.label); out += _sample(image, mask, ys, xs, patch, half, max(1, round(nb * scale)))
        rem = max(0, n_pos - len(out))
        if rem: ys, xs = np.where(mask > 0); out += _sample(image, mask, ys, xs, patch, half, rem)
    else:
        ys, xs = np.where(mask > 0); out += _sample(image, mask, ys, xs, patch, half, n_pos)
    return out
def hard_patches(image, mask, patch, n, sites, s):  # centred on current-detector false-positive sites, same jitter as _sample
    if sites is None or not len(sites): return random_patches(image, mask, patch, n)
    sc = np.asarray(sites) * s; return _sample(image, mask, sc[:, 0].astype(int), sc[:, 1].astype(int), patch, patch // 2, n)
def random_patches(image, mask, patch, n):
    H, W = mask.shape; out = []
    for _ in range(n):
        y0 = random.randint(0, max(0, H - patch)); x0 = random.randint(0, max(0, W - patch)); out.append((image[y0:y0 + patch, x0:x0 + patch], mask[y0:y0 + patch, x0:x0 + patch]))
    return out
class ImagePatches(Dataset):                         # one item = all patches of one (resized) photo
    def __init__(self, items, det):
        self.items, self.det, self.c = items.reset_index(drop=True), det, TRAIN_CFG[det]; self.aug = build_aug(det, self.c["patch"])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        r = self.items.iloc[i]; P = self.c["patch"]
        bgr = cv2.imread(str(ROOT / r["image"])); s = float(INFO.loc[r["name"], "s"]) if SCALED else 1.0   # r[...] not r.x: a Series row has methods such as .gt
        mask = (cv2.imread(str(ROOT / r["gt"]), cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8) * 255 if r["gt"] else np.zeros(bgr.shape[:2], np.uint8)
        img = preprocess(scale_img(bgr, s))
        if s > 1: mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        elif s < 1: mask = (cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_AREA) > 0).astype(np.uint8) * 255
        if r["kind"] == "positive": pts = lesion_patches(self.det, img, mask, P, self.c["n_pos"]) + random_patches(img, mask, P, self.c["n_neg"])
        elif NEG == "hard": pts = hard_patches(img, mask, P, self.c["n_neg"], HARD.get((self.det, r["name"])), s)
        else: pts = random_patches(img, mask, P, self.c["n_neg"])
        xs, ys = [], []
        for a, b in pts:
            if a.shape[:2] != (P, P): continue
            o = self.aug(image=a, mask=b); xs.append(o["image"]); ys.append(o["mask"] > 0)
        return (np.stack(xs), np.stack(ys)) if xs else (np.zeros((0, P, P, 3), np.uint8), np.zeros((0, P, P), bool))
def worker_init(wid):
    s = torch.initial_seed() % 2 ** 32; random.seed(s); np.random.seed(s)
def save_ckpt(model, path, **kw): torch.save(dict(state_dict=model.state_dict(), encoder="efficientnet-b2", **kw), str(path))
def backup(path, det):
    if not DRIVE_DIR: return
    r = subprocess.run(["rclone", "copy", str(path), f"{DRIVE_DIR}/{det}"], capture_output=True, text=True); print("  Drive backup", "ok" if r.returncode == 0 else f"FAILED: {r.stderr.strip()[:200]}", flush=True)
def qualifies(s, det):                               # selection rule; bounds are the RECORD validation values of the current detector
    rec, prec, fp = KNOWN_VAL[(det, "native")]
    return s[(s.healthy_fp_px <= fp) & (s.prec_tol_pct >= prec)]
def train(det):
    D = OUT / RUN(det); D.mkdir(exist_ok=True); LOG = D / "train_log.csv"; CK = Path(os.environ.get("RETRAIN_CKPT_DIR", "/root/retrain_ckpts")) / RUN(det) if DEVICE == "cuda" else D / "ckpts"; CK.mkdir(parents=True, exist_ok=True)
    (D / "run.json").write_text(json.dumps(dict(det=det, arm=ARM, seed=SEED, **ARMS[ARM])))
    if LOG.exists() and (D / "done.json").exists(): print(det, "already trained; delete", LOG, "to retrain"); return
    for f in (LOG, D / "val_by_epoch.csv", D / "done.json"): f.unlink(missing_ok=True)      # a restarted run starts clean
    set_seed(SEED); c = TRAIN_CFG[det]; rng = np.random.default_rng(SEED)
    model = smp.Unet(encoder_name="efficientnet-b2", encoder_weights="imagenet" if not SUBSET else None, in_channels=3, classes=1).to(DEVICE)
    ft, dice = FocalTverskyLoss(alpha=c["alpha"], beta=1 - c["alpha"]), DiceLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=c["wd"]); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP and DEVICE == "cuda")
    it = ITEMS[det]; fixed, healthy = it[it.kind != "healthy"], it[it.kind == "healthy"]
    logs, best, best_epoch, stale = [], -1.0, None, 0
    n_epochs = 1 if SUBSET else EPOCHS
    for epoch in range(1, n_epochs + 1):
        t0 = time.time(); model.train()
        items = pd.concat([fixed, healthy[rng.random(len(healthy)) < healthy.w.to_numpy()]])
        g = torch.Generator(); g.manual_seed(SEED * 1000 + epoch)
        dl = DataLoader(ImagePatches(items, det), batch_size=None, shuffle=True, num_workers=NUM_WORKERS, worker_init_fn=worker_init, generator=g, prefetch_factor=2 if NUM_WORKERS else None)
        tot, nb = 0.0, 0
        def step(x, y):
            x = torch.from_numpy(x).to(DEVICE).permute(0, 3, 1, 2).float().div(255); y = torch.from_numpy(y).to(DEVICE).float().unsqueeze(1)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP and DEVICE == "cuda"): logits = model(x)
            loss = ft(logits.float(), y) + dice(logits.float(), y); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); return float(loss)
        PX, PY = [], []                                  # as the old scripts: build every patch of the epoch, then shuffle them all together
        for xs, ys in dl: PX.append(xs.numpy()); PY.append(ys.numpy())
        X, Y = np.concatenate(PX), np.concatenate(PY); del PX, PY; npatch = len(X); perm = np.random.permutation(npatch)
        for j in range(0, npatch, c["batch"]):
            idx = np.sort(perm[j:j + c["batch"]]); tot += step(X[idx], Y[idx]); nb += 1
        del X, Y
        sched.step(); row = dict(epoch=epoch, loss=tot / max(nb, 1), patches=npatch, train_s=time.time() - t0)
        if epoch == 1 or epoch % VAL_EVERY == 0 or SUBSET:
            t1 = time.time(); df, _ = val_pass(model, det, scaled=SCALED); s = summary(df); s["epoch"] = epoch
            s.to_csv(D / "val_by_epoch.csv", mode="a", header=not (D / "val_by_epoch.csv").exists(), index=False)
            q = qualifies(s, det); score = float(q.recall_pct.max()) if len(q) else -1.0
            save_ckpt(model, CK / f"epoch{epoch:03d}.pth", epoch=epoch, det=det, seed=SEED, d_ref=D_REF)
            row.update(val_s=time.time() - t1, best_qualifying_recall=score)
            if score > best:
                best, best_epoch, stale = score, epoch, 0; save_ckpt(model, D / f"{det}_retrain_best.pth", epoch=epoch, det=det, seed=SEED, d_ref=D_REF); backup(D / f"{det}_retrain_best.pth", RUN(det))
            else: stale += 1
            print(f"{det} epoch {epoch:03d} loss {row['loss']:.4f} patches {npatch} train {row['train_s']:.0f}s val {row['val_s']:.0f}s | "
                  f"thr 0.35: recall {s.set_index('thr').loc[BASE_THR].recall_pct:.1f} prec {s.set_index('thr').loc[BASE_THR].prec_tol_pct:.1f} healthy fp {s.set_index('thr').loc[BASE_THR].healthy_fp_px:.1f} | best qualifying {best:.1f} (epoch {best_epoch})", flush=True)
        else: print(f"{det} epoch {epoch:03d} loss {row['loss']:.4f} patches {npatch} {row['train_s']:.0f}s", flush=True)
        logs.append(row); pd.DataFrame(logs).to_csv(LOG, index=False)
        if epoch >= MIN_EPOCHS and stale >= PATIENCE: print(det, "early stop at epoch", epoch); break
    (D / "done.json").write_text(json.dumps(dict(epochs_run=epoch, best_epoch=best_epoch, best_qualifying_recall=best, finished=time.strftime("%Y-%m-%d %H:%M"))))
    backup(LOG, RUN(det)); backup(D / "val_by_epoch.csv", RUN(det))
for det in DETS: train(det)

## 8. Selection on validation, learning curves and the three-path check on the chosen model

In [ ]:
SEL = {}
for det in DETS:
    D = OUT / RUN(det); V = pd.read_csv(D / "val_by_epoch.csv"); L = pd.read_csv(D / "train_log.csv")
    q = qualifies(V, det)
    best = q.sort_values(["recall_pct", "healthy_fp_px"], ascending=[False, True]).iloc[0] if len(q) else None
    SEL[det] = None if best is None else dict(epoch=int(best.epoch), thr=float(best.thr), recall_pct=best.recall_pct, prec_tol_pct=best.prec_tol_pct, healthy_fp_px=best.healthy_fp_px, healthy_any_pct=best.healthy_any_pct)
    print(det.upper(), "arm", ARM, "seed", SEED, "chosen:", SEL[det])
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
    ax[0].plot(L.epoch, L.loss, lw=1.2); ax[0].set(xlabel="epoch", ylabel="training loss", title=f"{det.upper()} arm {ARM} seed {SEED}: training loss")
    for t in (BASE_THR, 0.5):
        v = V[V.thr == t]; ax[1].plot(v.epoch, v.recall_pct, marker="o", ms=3, label=f"recall, thr {t}"); ax[1].plot(v.epoch, v.prec_tol_pct, marker="s", ms=3, ls="--", label=f"tolerant precision, thr {t}")
        ax[2].plot(v.epoch, v.healthy_fp_px, marker="o", ms=3, label=f"thr {t}")
    rec, prec, fp = KNOWN_VAL[(det, "native")]
    ax[1].axhline(rec, color="k", lw=0.8, ls=":", label="current detector recall"); ax[1].axhline(prec, color="grey", lw=0.8, ls=":", label="current detector precision")
    ax[2].axhline(fp, color="k", lw=0.8, ls=":", label="current detector"); ax[1].set(xlabel="epoch", ylabel="% (DDR validation half)"); ax[2].set(xlabel="epoch", ylabel="false-positive px per healthy eye (512)", yscale="symlog")
    for a in ax[1:]: a.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(D / "learning_curves.png", dpi=100); plt.show()
    if SEL[det]:
        ck = Path(os.environ.get("RETRAIN_CKPT_DIR", "/root/retrain_ckpts")) / RUN(det) / f"epoch{SEL[det]['epoch']:03d}.pth" if DEVICE == "cuda" else D / "ckpts" / f"epoch{SEL[det]['epoch']:03d}.pth"
        cand = smp.Unet(encoder_name="efficientnet-b2", encoder_weights=None, in_channels=3, classes=1); cand.load_state_dict(torch.load(ck, map_location=DEVICE, weights_only=False)["state_dict"]); cand.to(DEVICE).eval()
        save_ckpt(cand, D / f"{det}_retrain_selected.pth", epoch=SEL[det]["epoch"], thr=SEL[det]["thr"], det=det, arm=ARM, seed=SEED, scaled=SCALED, d_ref=D_REF); backup(D / f"{det}_retrain_selected.pth", RUN(det))
        df, kept = val_pass(cand, det, scaled=SCALED, keep=True); s = summary(df).set_index("thr").loc[SEL[det]["thr"]]
        assert abs(s.recall_pct - SEL[det]["recall_pct"]) <= 0.2 and abs(s.healthy_fp_px - SEL[det]["healthy_fp_px"]) <= max(1.0, 0.01 * SEL[det]["healthy_fp_px"]), ("rerun of the chosen checkpoint differs from the log", s.to_dict(), SEL[det])
        for n, M in kept.items():
            m = M[SEL[det]["thr"]]; a, fa = score_A(m, n, det); b, fb = score_B(m, n, det); c2, fc = score_C(m, n, det)
            assert a == b == c2 and np.array_equal(fa, fb) and np.array_equal(fa, fc), (det, n, a, b, c2)
        print(det.upper(), "chosen checkpoint re-run reproduces the log; three matching paths agree on all", len(kept), "validation images")
        SEL[det]["md5"] = hashlib.md5((D / f"{det}_retrain_selected.pth").read_bytes()).hexdigest()
(OUT / f"selection_{TAG}.json").write_text(json.dumps(SEL, indent=1, default=float))

## 9. Confirmation on the test data (once; `RETRAIN_CONFIRM=1`)
Written before any test number existed. Current detector (production configuration, threshold 0.35) vs the retrained one (resized photos, threshold chosen on validation) on the same test images: DDR-test test half, IDRiD-test, 300 healthy holdout eyes. The current detector's numbers must first reproduce the Stage 1 record. Tests per detector (Holm across that detector's family): lesion-level McNemar (exact) on DDR and IDRiD; per-image Wilcoxon on false-positive px (DDR, IDRiD, healthy); McNemar on healthy eyes with any detection.

**Adoption rule.** The retrained detector is recommended only if no test shows a significant worsening (fewer lesions found, more false-positive px, more healthy eyes flagged) and at least one shows a significant improvement (Holm-adjusted p < 0.05). Otherwise the current detector stays.

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
def wilson(k, n): ci = stats.binomtest(int(k), int(n)).proportion_ci(0.95, method="wilson"); return 100 * ci.low, 100 * ci.high
def boot_ratio(df, num, den, rng):
    a = df[[num, den]].to_numpy(float); idx = rng.integers(0, len(a), (N_BOOT, len(a))); s = a[idx].sum(1)
    with np.errstate(invalid="ignore", divide="ignore"): return np.nanpercentile(100 * s[:, 0] / s[:, 1], [2.5, 97.5])
def mcnemar(a, b):                                   # paired booleans (current, retrained); exact two-sided; change = retrained minus current
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum()); return (stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0), n01 - n10
def wilcox(x, y):
    d = np.asarray(y, float) - np.asarray(x, float)
    return (stats.wilcoxon(x, y).pvalue if np.any(d != 0) else 1.0), float(np.median(d))
CONF_RES, CONF_TESTS = [], []
if CONFIRM and not SUBSET:
    TESTM = man[man.split == "test"].reset_index(drop=True); load_cache(TESTM); rng = np.random.default_rng(0)
    for det in DETS:
        if not SEL.get(det): print(det.upper(), "no candidate chosen on validation: nothing to confirm"); continue
        D = OUT / RUN(det); prod = load_prod(det); thr = SEL[det]["thr"]; sp = D / f"{det}_retrain_selected.pth"
        assert hashlib.md5(sp.read_bytes()).hexdigest() == SEL[det]["md5"], "selected checkpoint changed since selection"
        cand = smp.Unet(encoder_name="efficientnet-b2", encoder_weights=None, in_channels=3, classes=1)
        cand.load_state_dict(torch.load(sp, map_location=DEVICE, weights_only=False)["state_dict"]); cand.to(DEVICE).eval()
        rows, flags = [], {}
        for i, r in enumerate(TESTM.itertuples()):
            bgr = cv2.imread(str(ROOT / r.image))
            for arm, m in (("current", masks512(prob_map(prod, bgr, det, False, r.name), r.name, det, [BASE_THR])[BASE_THR]),
                           ("retrained", masks512(prob_map(cand, bgr, det, SCALED, r.name), r.name, det, [thr])[thr])):
                a, fa = score_A(m, r.name, det); b, fb = score_B(m, r.name, det); c2, fc = score_C(m, r.name, det)
                assert a == b == c2 and np.array_equal(fa, fb) and np.array_equal(fa, fc), (det, r.name, arm)
                rows.append(dict(name=r.name, group=r.group, arm=arm, **a)); flags[(arm, r.name)] = fa
            if i % 50 == 0: print(det, "test", i + 1, "/", len(TESTM), flush=True)
        T = pd.DataFrame(rows); T.to_csv(D / "test_per_image.csv", index=False)
        for g in ("DDR", "IDRiD", "healthy"):
            for arm in ("current", "retrained"):
                d = T[(T.group == g) & (T.arm == arm)]
                if g == "healthy":
                    k = int((d.pred_px > 0).sum()); lo, hi = wilson(k, len(d)); b = d.fp_px.to_numpy(float); bi = rng.integers(0, len(b), (N_BOOT, len(b)))
                    flo, fhi = np.percentile(b[bi].mean(1), [2.5, 97.5])
                    CONF_RES.append(dict(det=det, group=g, arm=arm, images=len(d), fp_px_per_eye=b.mean(), fp_ci_low=flo, fp_ci_high=fhi, any_pct=100 * k / len(d), any_ci_low=lo, any_ci_high=hi))
                else:
                    lo, hi = boot_ratio(d[d.lesions > 0], "found", "lesions", rng)
                    CONF_RES.append(dict(det=det, group=g, arm=arm, images=int((d.lesions > 0).sum()), lesions=int(d.lesions.sum()), recall_pct=100 * d.found.sum() / d.lesions.sum(),
                                         recall_ci_low=lo, recall_ci_high=hi, prec_tol_pct=100 * d.tp_tol.sum() / max(d.pred_px.sum(), 1), fp_px_per_image=d.fp_px.mean()))
        R = pd.DataFrame([x for x in CONF_RES if x["det"] == det]); display(R.round(2))
        for g in ("DDR", "IDRiD", "healthy"):
            cur = R[(R.group == g) & (R.arm == "current")].iloc[0]; k1, k2 = KNOWN_TEST[(det, g)]
            ok = (abs(cur.fp_px_per_eye - k1) <= max(2.0, 0.02 * k1) and abs(cur.any_pct - k2) <= 0.5) if g == "healthy" else (abs(cur.recall_pct - k1) <= 0.3 and abs(cur.prec_tol_pct - k2) <= 0.3)
            assert ok, f"{det} {g}: current detector does not reproduce the Stage 1 test record: {cur.to_dict()}"
        print(det.upper(), "current detector reproduces the Stage 1 test record")
        fam = []
        for g in ("DDR", "IDRiD"):
            nm = TESTM[TESTM.group == g].name
            a = np.concatenate([flags[("current", n)] for n in nm]); b = np.concatenate([flags[("retrained", n)] for n in nm])
            p, diff = mcnemar(a, b); fam.append(dict(det=det, group=g, test="lesions found (McNemar)", change=diff, p=p, worse=diff < 0))
        for g in ("DDR", "IDRiD", "healthy"):
            x = T[(T.group == g) & (T.arm == "current")].sort_values("name").fp_px.to_numpy(); y = T[(T.group == g) & (T.arm == "retrained")].sort_values("name").fp_px.to_numpy()
            p, med = wilcox(x, y); fam.append(dict(det=det, group=g, test="false-positive px per image (Wilcoxon)", change=float(y.mean() - x.mean()), median_change=med, p=p, worse=y.mean() > x.mean()))
        hp = T[T.group == "healthy"].pivot(index="name", columns="arm", values="pred_px") > 0
        p, diff = mcnemar(hp["current"].to_numpy(), hp["retrained"].to_numpy()); fam.append(dict(det=det, group="healthy", test="eyes with any detection (McNemar)", change=diff, p=p, worse=diff > 0))
        F = pd.DataFrame(fam); F["p_holm"] = holm(F.p); F["significant"] = F.p_holm < 0.05
        F["verdict"] = np.where(~F.significant, "no detectable change", np.where(F.worse, "worse", "better")); display(F.round(4)); CONF_TESTS.append(F)
        SEL[det]["adopt"] = bool(not (F.significant & F.worse).any() and (F.significant & ~F.worse).any())
        print(det.upper(), "adoption rule:", "RECOMMEND the retrained detector" if SEL[det]["adopt"] else "keep the current detector")
    if CONF_RES:
        pd.DataFrame(CONF_RES).to_csv(OUT / f"test_results_{TAG}.csv", index=False); pd.concat(CONF_TESTS).to_csv(OUT / f"test_tests_{TAG}.csv", index=False)
        (OUT / f"selection_{TAG}.json").write_text(json.dumps(SEL, indent=1, default=float))
else: print("confirmation not run (set RETRAIN_CONFIRM=1 after the detector is chosen)")

## 10. Claims (the only source for messages)

In [ ]:
claims = []
def add(kind, text, evidence): claims.append(dict(kind=kind, claim=text, evidence=evidence))
if not SUBSET:
    for r in KNOWN.itertuples():
        add("measured", f"{r.det.upper()} current detector on validation ({r.mode}): recall {rhu(r.recall_now, 1)}% (record {rhu(r.recall_record, 1)}), tolerant precision {rhu(r.prec_now, 1)}% (record {rhu(r.prec_record, 1)}), {rhu(r.fp_now, 1)} false-positive px per healthy eye (record {rhu(r.fp_record, 1)}).", "Section 4")
    for det in DETS:
        sc = SCREEN[det]; it = ITEMS[det]
        add("measured", f"{det.upper()} blank-mask screen: {int(sc.excluded.sum())} of {len(sc)} blank-mask DDR training images left out (detected area above {rhu(sc.cutoff.iloc[0], 1)} px at 512, the 95th percentile of validation healthy eyes).", "Section 5")
        add("measured", f"{det.upper()} training images: {int((it.kind == 'positive').sum())} with lesions, {int((it.kind == 'outlined negative').sum())} outlined negatives, {int((it.kind == 'healthy').sum())} healthy (each epoch about half).", "Section 6")
        if det in SEL and SEL[det]:
            s = SEL[det]; rec, prec, fp = KNOWN_VAL[(det, "native")]
            add("measured", f"{det.upper()} retrained (arm {ARM}, seed {SEED}), chosen on validation: epoch {s['epoch']}, threshold {s['thr']}; DDR-val recall {rhu(s['recall_pct'], 1)}% vs current {rhu(rec, 1)}%; tolerant precision {rhu(s['prec_tol_pct'], 1)}% vs {rhu(prec, 1)}%; {rhu(s['healthy_fp_px'], 1)} false-positive px per healthy eye vs {rhu(fp, 1)}.", "Section 8")
        elif det in SEL:
            add("measured", f"{det.upper()} retrained (arm {ARM}, seed {SEED}): no epoch and threshold met both validation bounds (healthy false positives and precision no worse than the current detector).", "Section 8")
    for x in CONF_RES:
        if x["group"] == "healthy":
            add("measured", f"{x['det'].upper()} {x['arm']} detector, test healthy eyes ({x['images']}): {rhu(x['fp_px_per_eye'], 1)} false-positive px per eye (95% CI {rhu(x['fp_ci_low'], 1)}-{rhu(x['fp_ci_high'], 1)}); something flagged in {rhu(x['any_pct'])}% ({rhu(x['any_ci_low'])}-{rhu(x['any_ci_high'])}).", "Section 9")
        else:
            add("measured", f"{x['det'].upper()} {x['arm']} detector, test {x['group']}: {rhu(x['recall_pct'])}% of {x['lesions']} grader lesions found (95% CI {rhu(x['recall_ci_low'])}-{rhu(x['recall_ci_high'])}); tolerant precision {rhu(x['prec_tol_pct'])}%.", "Section 9")
    for F in CONF_TESTS:
        for r in F.itertuples(): add("measured", f"{r.det.upper()} test, {r.group}, {r.test}: {r.verdict} (Holm p {rhu(r.p_holm, 3)}).", "Section 9")
    for det in DETS:
        if SEL.get(det) and "adopt" in SEL[det]: add("measured", f"{det.upper()}: the pre-specified adoption rule says {'recommend the retrained detector' if SEL[det]['adopt'] else 'keep the current detector'}.", "Section 9")
    if not CONF_RES: add("untested", "Validation results are not test results: the one test comparison (section 9) decides whether a retrained detector is better.", "Design")
CL = pd.DataFrame(claims); (OUT / f"claims_{TAG}.csv").write_text(CL.to_csv(index=False)) if len(CL) else None
display(CL)